## Load libraries

In [19]:
import os
import time
import random
import numpy as np
import pandas as pd
import torch
import itertools, random

from sklearn.metrics import mean_absolute_error, mean_squared_error
import lightning.pytorch as pl
from lightning.pytorch import Trainer
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer, QuantileLoss
from pytorch_forecasting.data import GroupNormalizer
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_forecasting import (
    TimeSeriesDataSet,
    TemporalFusionTransformer,
    QuantileLoss
)

In [7]:
torch.set_float32_matmul_precision("high")

print("Lightning version:", pl.__version__)
print("TFT subclass of LightningModule:", issubclass(TemporalFusionTransformer, pl.LightningModule))
print(torch.get_float32_matmul_precision())


Lightning version: 2.6.0
TFT subclass of LightningModule: True
high


## Config

In [16]:

DATA_PATH = "../../data/full_data.xlsx"
OUTPUT_CSV = "../../results/tft_tuning_results.csv"

TIME_COL   = "Date"
TARGET_COL = "AveragePrice"
ENTITY_COL = "AreaCode"

TRAIN_START_DATE = pd.Timestamp("2007-04-01")
TRAIN_END_DATE = pd.Timestamp("2020-12-31")
VAL_END_DATE   = pd.Timestamp("2022-03-31")

BATCH_SIZE = 64
MAX_EPOCHS = 1
PATIENCE   = 8
NUM_WORKERS = 4
SEED = 42

WINDOW_CHOICES = [12, 18]   # encoder length
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)


## Metrics

In [9]:
def mae(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return float(np.mean(np.abs(y - yhat)))

def rmse(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return float(np.sqrt(np.mean((y - yhat) ** 2)))

def smape(y, yhat, eps=1e-8):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return 100.0 * np.mean(2.0 * np.abs(yhat - y) / (np.abs(y) + np.abs(yhat) + eps))

def mase(y_true, y_pred, y_train, m=12, eps=1e-8):
    y_train = np.asarray(y_train)
    if len(y_train) <= m:
        return np.nan
    scale = np.mean(np.abs(y_train[m:] - y_train[:-m])) + eps
    return np.mean(np.abs(y_true - y_pred)) / scale

## Load data

In [10]:

df = pd.read_excel(DATA_PATH, parse_dates=[TIME_COL])
df = df.sort_values([ENTITY_COL, TIME_COL]).reset_index(drop=True)

df["time_idx"] = (
    (df[TIME_COL].dt.year - df[TIME_COL].dt.year.min()) * 12
    + df[TIME_COL].dt.month - 1
)

# restrict to period with full feature availability
mask_train = (df[TIME_COL] >= TRAIN_START_DATE) & (df[TIME_COL] <= TRAIN_END_DATE)
mask_val   = (df[TIME_COL] > TRAIN_END_DATE)    & (df[TIME_COL] <= VAL_END_DATE)

df_train = df.loc[mask_train].copy()
df_val   = df.loc[mask_val].copy()

known_las = df_train[ENTITY_COL].unique()

df_val = df_val[df_val[ENTITY_COL].isin(known_las)].copy()


# -------- Feature lists (ADJUST TO YOUR DATA) --------
continuous_cols = [
    "AverageNeighbourPrice", "local_I", "area_km2",
    "centroid_x", "centroid_y", "CoL_distance_km",
    "LA_FE", "sdlt_perc_threshold", "dwelling_stock",
    "population", "ashe_weekly", "base_rate",
    "claimant_count_prop", "planning_decisions_per_1000",
    "planning_granted_prop", "rail_station_entry_exit",
    "GDP", "CPIH",

    
    "LMIQuadrant__2", "LMIQuadrant__3", "LMIQuadrant__4",
    "Region_East of England", "Region_London",
    "Region_North East", "Region_North West",
    "Region_South East", "Region_South West",
    "Region_West Midlands", "Region_Yorkshire and The Humber"
]

categorical_cols = [
]

value_cols = continuous_cols + [TARGET_COL]

# 1) forward/backward fill within each LA over time
for col in value_cols:
    df_train[col] = (
        df_train
        .groupby(ENTITY_COL)[col]
        .ffill()
        .bfill()
    )
    df_val[col] = (
        df_val
        .groupby(ENTITY_COL)[col]
        .ffill()
        .bfill()
    )

# 2) if anything is still missing (e.g. weird LAs), fill with global column mean
for col in value_cols:
    if df_train[col].isna().any() or df_val[col].isna().any():
        col_mean = df_train[col].mean()
        df_train[col] = df_train[col].fillna(col_mean)
        df_val[col]   = df_val[col].fillna(col_mean)

print("NaNs in df_train[continuous_cols]:", df_train[continuous_cols].isna().sum().sum())
print("NaNs in df_val[continuous_cols]:", df_val[continuous_cols].isna().sum().sum())

print(len(df_train[ENTITY_COL].unique()),
      len(df_val[ENTITY_COL].unique()))

df_train[ENTITY_COL] = df_train[ENTITY_COL].astype(str)
df_val[ENTITY_COL]   = df_val[ENTITY_COL].astype(str)

# === restrict val to LAs that exist in train ===
train_las = set(df_train[ENTITY_COL].unique())
val_las   = set(df_val[ENTITY_COL].unique())

print("Train LA count:", len(train_las))
print("Val LA count:", len(val_las))
print("Unseen in val before filtering:", val_las - train_las)

df_val = df_val[df_val[ENTITY_COL].isin(train_las)].copy()

val_las_after = set(df_val[ENTITY_COL].unique())
print("Unseen in val after filtering:", val_las_after - train_las)
print("Final train/val LA counts:",
      len(train_las), len(val_las_after))

NaNs in df_train[continuous_cols]: 0
NaNs in df_val[continuous_cols]: 0
294 294
Train LA count: 294
Val LA count: 294
Unseen in val before filtering: set()
Unseen in val after filtering: set()
Final train/val LA counts: 294 294


## Hyperparameters

In [11]:
param_space = {
    "WINDOW": WINDOW_CHOICES,
    "hidden_size": [32, 48],
    "attention_head_size": [2, 4],
    "dropout": [0.05, 0.10, 0.20],
    "learning_rate": [1e-3, 5e-4, 2e-4],
}

fixed_configs = [
    {"WINDOW": 12, "hidden_size": 32, "attention_head_size": 4, "dropout": 0.1, "learning_rate": 1e-3},
    {"WINDOW": 18, "hidden_size": 48, "attention_head_size": 4, "dropout": 0.1, "learning_rate": 5e-4}
]

# 1) build full grid
keys = list(param_space.keys())
all_grid = [
    dict(zip(keys, values))
    for values in itertools.product(*(param_space[k] for k in keys))
]

# 2) make a set of fixed configs (hashable representation)
def cfg_key(cfg):
    return tuple((k, cfg[k]) for k in sorted(cfg.keys()))

fixed_keys = {cfg_key(cfg) for cfg in fixed_configs}

# 3) filter out fixed configs from the grid
available_for_random = [
    cfg for cfg in all_grid
    if cfg_key(cfg) not in fixed_keys
]

# 4) sample N random configs from the remaining ones
N_RANDOM = 3  # or whatever you want
random_configs = random.sample(available_for_random, N_RANDOM)

# 5) final list with no overlap
tuning_configs = fixed_configs + random_configs


In [12]:
df_train[TARGET_COL] = df_train[TARGET_COL].astype(float)
df_val[TARGET_COL]   = df_val[TARGET_COL].astype(float)

## Tuning

In [29]:

# ============================================================
# TUNING LOOP
# ============================================================

results = []

for run_id, cfg in enumerate(tuning_configs, start=1):

    print(f"\n=== TFT CONFIG {run_id} / {len(tuning_configs)} ===")
    print(cfg)

    start_time = time.time()

    training_ds = TimeSeriesDataSet(
        df_train,
        time_idx="time_idx",
        target=TARGET_COL,
        group_ids=[ENTITY_COL],   # only categorical allowed
        max_encoder_length=cfg["WINDOW"],
        max_prediction_length=1,

        time_varying_known_reals=continuous_cols,
        time_varying_unknown_reals=[TARGET_COL],

        time_varying_known_categoricals=[],
        time_varying_unknown_categoricals=[],
        static_categoricals=[],
        static_reals=[],

        # ❌ remove target_normalizer=GroupNormalizer(...)
        add_relative_time_idx=True,
        add_target_scales=True,   # ✅ let PF handle scaling internally
        add_encoder_length=True,
        allow_missing_timesteps=True,
    )

    print("TFT categorical encoders:")
    print(training_ds.categorical_encoders)


    val_ds = TimeSeriesDataSet.from_dataset(
        training_ds,
        df_val,
        predict=True,
        stop_randomization=True,
        allow_missing_timesteps=True
    )

    train_loader = training_ds.to_dataloader(
        train=True, batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS, persistent_workers=True
    )

    val_loader = val_ds.to_dataloader(
        train=False, batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS, persistent_workers=True
    )

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=PATIENCE,
        mode="min"
    )

    tft = TemporalFusionTransformer.from_dataset(
        training_ds,
        hidden_size=cfg["hidden_size"],
        attention_head_size=cfg["attention_head_size"],
        dropout=cfg["dropout"],
        learning_rate=cfg["learning_rate"],
        loss=QuantileLoss([0.5]),
        optimizer="Adam",
        reduce_on_plateau_patience=4,
    )

    trainer = Trainer(
        max_epochs=MAX_EPOCHS,
        accelerator="gpu" if torch.cuda.is_available() else "cpu",
        devices=1,
        # precision="16-mixed",
        callbacks=[early_stop],
        enable_checkpointing=False,
        logger=False,
        enable_model_summary=False,
    )

    trainer.fit(tft, train_loader, val_loader)

    # =======================
    # VALIDATION EVALUATION
    # =======================

    # =======================
# VALIDATION EVALUATION
# =======================

tft.eval()
y_true_list = []
y_pred_list = []

device = next(tft.parameters()).device

with torch.no_grad():
    for batch in val_loader:
        # batch can be (x, y) or (x, (y, weight))
        if isinstance(batch, (list, tuple)):
            x = batch[0]
            y = batch[1]
        else:
            x, y = batch

        # x is a dict of tensors
        x = {k: v.to(device) for k, v in x.items()}

        # y may itself be a (target, weight) tuple → take the target
        if isinstance(y, (list, tuple)):
            y = y[0]

        y = y.to(device)  # now definitely a tensor

        out = tft(x)  # [batch, pred_len] or [batch, 1, ...]

        # we set max_prediction_length=1, so take the last prediction step
        if out.ndim == 3:          # [B, pred_len, 1]
            y_hat = out[:, -1, 0]
        elif out.ndim == 2:        # [B, pred_len]
            y_hat = out[:, -1]
        else:                      # fallback
            y_hat = out.squeeze()

        if y.ndim == 3:            # [B, pred_len, 1]
            y_step = y[:, -1, 0]
        elif y.ndim == 2:          # [B, pred_len]
            y_step = y[:, -1]
        else:
            y_step = y.squeeze()

        y_pred_list.append(y_hat.detach().cpu().numpy())
        y_true_list.append(y_step.detach().cpu().numpy())


    y_pred = np.concatenate(y_pred_list)
    y_true = np.concatenate(y_true_list)

    y_train_full = df_train[TARGET_COL].values

    mae_val  = mean_absolute_error(y_true, y_pred)
    rmse_val = np.sqrt(mean_squared_error(y_true, y_pred))
    smape_val = smape(y_true, y_pred)
    mase_val  = mase(y_true, y_pred, y_train_full)


    elapsed = time.time() - start_time

    results.append({
        "model_type": "TFT",
        "WINDOW": cfg["WINDOW"],
        "hidden_size": cfg["hidden_size"],
        "attention_heads": cfg["attention_head_size"],
        "dropout": cfg["dropout"],
        "learning_rate": cfg["learning_rate"],
        "MAE": mae_val,
        "RMSE": rmse_val,
        "sMAPE": smape_val,
        "MASE": mase_val,
        "train_time_sec": elapsed

    })

    print(f"MAE={mae_val:.2f}  RMSE={rmse_val:.2f}  sMAPE={smape_val:.3f}  MASE={mase_val:.3f}")
    print(f"Time: {elapsed/60:.1f} min")

# ============================================================
# SAVE RESULTS
# ============================================================

results_df = pd.DataFrame(results)
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
results_df.to_csv(OUTPUT_CSV, index=False)

print("\n✅ TFT tuning complete. Results saved to:")
print(OUTPUT_CSV)



=== TFT CONFIG 1 / 5 ===
{'WINDOW': 12, 'hidden_size': 32, 'attention_head_size': 4, 'dropout': 0.1, 'learning_rate': 0.001}


c:\Users\slong\Documents\MSc code\UK-average-house-price-prediction\.venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:210: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
c:\Users\slong\Documents\MSc code\UK-average-house-price-prediction\.venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:210: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


TFT categorical encoders:
{'__group_id__AreaCode': NaNLabelEncoder(add_nan=False, warn=True), 'AreaCode': NaNLabelEncoder(add_nan=False, warn=True)}
Sanity Checking: |          | 0/? [00:00<?, ?it/s]

c:\Users\slong\Documents\MSc code\UK-average-house-price-prediction\.venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Epoch 0: 100%|██████████| 1405/1405 [03:14<00:00,  7.24it/s, train_loss_step=5.25e+3, val_loss=1.38e+4, train_loss_epoch=9.89e+3]

=== TFT CONFIG 2 / 5 ===
{'WINDOW': 18, 'hidden_size': 48, 'attention_head_size': 4, 'dropout': 0.1, 'learning_rate': 0.0005}
TFT categorical encoders:
{'__group_id__AreaCode': NaNLabelEncoder(add_nan=False, warn=True), 'AreaCode': NaNLabelEncoder(add_nan=False, warn=True)}


c:\Users\slong\Documents\MSc code\UK-average-house-price-prediction\.venv\Lib\site-packages\pytorch_forecasting\data\timeseries.py:1282: UserWarning: Min encoder length and/or min_prediction_idx and/or min prediction length and/or lags are too large for 294 series/groups which therefore are not present in the dataset index. This means no predictions can be made for those series. First 10 removed groups: [{'__group_id__AreaCode': 'E06000001'}, {'__group_id__AreaCode': 'E06000002'}, {'__group_id__AreaCode': 'E06000003'}, {'__group_id__AreaCode': 'E06000004'}, {'__group_id__AreaCode': 'E06000005'}, {'__group_id__AreaCode': 'E06000006'}, {'__group_id__AreaCode': 'E06000007'}, {'__group_id__AreaCode': 'E06000008'}, {'__group_id__AreaCode': 'E06000009'}, {'__group_id__AreaCode': 'E06000010'}]
  warnings.warn(


AssertionError: filters should not remove entries all entries - check encoder/decoder lengths and lags

In [ ]:
for col in df_val.columns:
    if df_val[col].astype(str).eq("106548").any():
        print("106548 found in column:", col, "dtype:", df_val[col].dtype)

106548 found in column: AveragePrice dtype: int64
